# Ensemble & Final

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

oof = pd.read_csv('oof_predictions.csv')
test_pred = pd.read_csv('test_predictions.csv')

model_cols = ['logreg','xgb','lgbm','cb']
tree_cols = ['xgb','lgbm','cb']
y = oof['Transported'].astype(int)

oof.head()

,PassengerId,Transported,logreg,xgb,lgbm,cb
0,0001_01,0,0.845018,0.748954,0.713303,0.735688
1,0002_01,1,0.165262,0.175652,0.130253,0.231241
2,0003_01,0,0.000019,0.074496,0.005612,0.036057
3,0003_02,0,0.006809,0.023398,0.008593,0.018719
4,0004_01,1,0.155082,0.123436,0.090743,0.211043


### Корреляция предсказаний моделей

In [2]:
oof[model_cols].corr().round(3)

,logreg,xgb,lgbm,cb
logreg,1.000,0.934,0.933,0.932
xgb,0.934,1.000,0.991,0.989
lgbm,0.933,0.991,1.000,0.988
cb,0.932,0.989,0.988,1.000


`xgb`/`lgbm`/`cb` коррелируют между собой на 0.986-0.991 — почти одно и то же,
диверсити между бустингами минимальная. `logreg` отличается заметно сильнее
(0.93-0.94), но и качество у неё несколько ниже (0.7963 против 0.81+ у бустингов).

### Voting

In [3]:
voting_all = oof[model_cols].mean(axis=1)
voting_trees = oof[tree_cols].mean(axis=1)

accuracy_score(y, voting_all > 0.5), accuracy_score(y, voting_trees > 0.5)

(0.8101921085931209, 0.8135281260784539)

### Weighted voting

In [4]:
scores = {c: accuracy_score(y, oof[c] > 0.5) for c in tree_cols}
weights = np.array([scores[c] - 0.5 for c in tree_cols])
weights = weights / weights.sum()

weighted_trees = (oof[tree_cols] * weights).sum(axis=1)
dict(zip(tree_cols, weights.round(3))), accuracy_score(y, weighted_trees > 0.5)

({'xgb': np.float64(0.333),
  'lgbm': np.float64(0.333),
  'cb': np.float64(0.334)},
 0.8135281260784539)

Веса получились практически равными (0.333/0.333/0.334) — модели настолько близки
по качеству, что взвешивание по CV score почти не отличается от простого
усреднения (0.8127 в обоих случаях).

### Stacking

Мета-модель — LogisticRegression поверх OOF-предсказаний базовых моделей.
Раз OOF уже честные (out-of-fold), оцениваем мета-модель дополнительным
StratifiedKFold(5) поверх них — так же, как валидировали бы обычную модель.

In [5]:
def stacking_cv(feature_cols):
    X = oof[feature_cols].values
    y_arr = y.values
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    meta_oof = np.zeros(len(X))
    for tr_idx, val_idx in skf.split(X, y_arr):
        meta = LogisticRegression()
        meta.fit(X[tr_idx], y_arr[tr_idx])
        meta_oof[val_idx] = meta.predict_proba(X[val_idx])[:, 1]
    return meta_oof

stacking_all4 = stacking_cv(model_cols)
stacking_trees = stacking_cv(tree_cols)

accuracy_score(y, stacking_all4 > 0.5), accuracy_score(y, stacking_trees > 0.5)

(0.8130679857356494, 0.813643161164155)

Стекинг по трём бустингам — лучший результат из всех подходов: 0.8130.
Мета-модель на всех 4 моделях (0.8127) чуть хуже — она сама учится
занижать вес `logreg`, но не полностью его игнорирует, в отличие от
варианта, где мы просто не даём ей `logreg` на вход.

### Сравнение всех подходов

In [6]:
pd.DataFrame({
    'strategy': ['logreg', 'xgb', 'lgbm', 'cb',
                 'voting_all4', 'voting_trees', 'weighted_trees',
                 'stacking_all4', 'stacking_trees'],
    'cv_accuracy': [
        accuracy_score(y, oof['logreg'] > 0.5),
        accuracy_score(y, oof['xgb'] > 0.5),
        accuracy_score(y, oof['lgbm'] > 0.5),
        accuracy_score(y, oof['cb'] > 0.5),
        accuracy_score(y, voting_all > 0.5),
        accuracy_score(y, voting_trees > 0.5),
        accuracy_score(y, weighted_trees > 0.5),
        accuracy_score(y, stacking_all4 > 0.5),
        accuracy_score(y, stacking_trees > 0.5),
    ],
}).sort_values('cv_accuracy', ascending=False)

,strategy,cv_accuracy
8,stacking_trees,0.813643
6,weighted_trees,0.813528
5,voting_trees,0.813528
7,stacking_all4,0.813068
3,cb,0.812608
2,lgbm,0.811688
1,xgb,0.811112
4,voting_all4,0.810192
0,logreg,0.796273


`stacking_trees` — лучший вариант (0.8130), берём его для финального submission.
Прирост над лучшей одиночной моделью (CatBoost, 0.8126) минимальный —
ожидаемо, учитывая как сильно коррелируют бустинги между собой. Ансамбль
дал больше пользы на этапе voting/stacking именно за счёт исключения слабой
`logreg`, чем за счёт объединения разных моделей.

### Финальный submission

Обучаем мета-модель на всех OOF (трёх бустингов), применяем к test-предсказаниям


In [7]:
final_meta = LogisticRegression()
final_meta.fit(oof[tree_cols], y)

test_proba = final_meta.predict_proba(test_pred[tree_cols])[:, 1]

submission = pd.DataFrame({
    'PassengerId': test_pred['PassengerId'],
    'Transported': test_proba > 0.5,
})

submission.to_csv('submission_final.csv', index=False)
submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [8]:
submission['Transported'].value_counts(normalize=True).round(4)

Transported
True     0.534
False    0.466
Name: proportion, dtype: float64

Баланс предсказанных классов на test (53.4%/46.6%) немного смещён в сторону True относительно train (50.4%/49.6%)

### Итог по пайплайну

| Этап | CV accuracy |
|---|---|
| baseline (`02`) | 0.7853 |
| + Feature Engineering (`03`, LogReg) | 0.7963 |
| лучшая одиночная модель (`04`, CatBoost) | 0.8126 |
| финальный ансамбль (`05`, stacking по 3 бустингам) | 0.8130 |

Итоговый прирост от baseline: **+2.77 п.п.** (0.7853 → 0.8130). Большая часть
прироста — от Feature Engineering и перехода на бустинги, ансамбль добавил
немного поверх уже сильных и похожих друг на друга моделей. Но в итоге даже отдельные бустинги существенно снижают интерпретируемость модели. В рамках данного дата-сета это очевидно не критично. Но в реальном бизнесе в подобной ситуации, видимо, лучше было остаться на простом понятном baseline